In [1]:

# This script demonstrates the core logic of gradient boosting for a regression
# problem. We will manually build a model step-by-step without using a
# pre-built library function, showing each round explicitly.
#
# The goal is to predict house prices based on their size.

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# --- 0. Prepare a Simple Dataset ---
# Let's create a small dataset of 10 houses.
# Feature: 'Size' in square feet.
# Target: 'Price' in thousands of dollars.
data = {
    'Size': [1400, 1600, 1700, 1875, 1100, 1550, 2350, 2450, 1425, 1700],
    'Price': [245, 312, 279, 308, 199, 219, 405, 324, 319, 255]
}
df = pd.DataFrame(data)

X = df[['Size']]
y = df['Price']

print("--- Initial Dataset ---")
print(df)
print("-" * 30, "\n")


# --- Boosting Hyperparameters ---
LEARNING_RATE = 0.8 # Eta (η) - how much each weak learner contributes



# Step 1: Initialize the model with a constant value (F_0).
# The first prediction is simply the mean of the target variable.
initial_prediction = y.mean()
df['F_0'] = initial_prediction

print(f"--- Round 0: Initial Prediction (F_0) ---")
print(f"The first prediction for all houses is the mean price: {initial_prediction:.2f}")

# Step 2: Compute residuals for the initial model.
# Residuals = Actual Price - Predicted Price
df['Residual_0'] = y - df['F_0']
print("\nStep 2: Compute initial residuals (y - F_0)")
print(df[['Size', 'Price', 'F_0', 'Residual_0']].round(2))
initial_mse = mean_squared_error(y, df['F_0'])
print(f"\nMean Squared Error (MSE) for initial model: {initial_mse:.2f}")
print("-" * 50, "\n")


--- Initial Dataset ---
   Size  Price
0  1400    245
1  1600    312
2  1700    279
3  1875    308
4  1100    199
5  1550    219
6  2350    405
7  2450    324
8  1425    319
9  1700    255
------------------------------ 

--- Round 0: Initial Prediction (F_0) ---
The first prediction for all houses is the mean price: 286.50

Step 2: Compute initial residuals (y - F_0)
   Size  Price    F_0  Residual_0
0  1400    245  286.5       -41.5
1  1600    312  286.5        25.5
2  1700    279  286.5        -7.5
3  1875    308  286.5        21.5
4  1100    199  286.5       -87.5
5  1550    219  286.5       -67.5
6  2350    405  286.5       118.5
7  2450    324  286.5        37.5
8  1425    319  286.5        32.5
9  1700    255  286.5       -31.5

Mean Squared Error (MSE) for initial model: 3260.05
-------------------------------------------------- 



In [2]:


# ####################################################################
# --- Round 1: Adding the First Weak Learner (h_1) ---
# ####################################################################
print(f"--- Round 1: Adding Weak Learner 1 (h_1) ---")

# Step 3: Fit a weak learner (a decision stump) to the residuals from Round 0.
print(f"\nStep 3: Fit a decision stump (h_1) to the residuals from Round 0.")
stump_1 = DecisionTreeRegressor(max_depth=1, random_state=42)
stump_1.fit(X, df['Residual_0'])

# Predict the error corrections using the new stump
df['h_1'] = stump_1.predict(X)
print("The stump has learned to predict the following error corrections:")
print(df[['h_1']].round(2))


# Step 4: Update the model's predictions to F_1.
# F_1(x) = F_0(x) + learning_rate * h_1(x)
df['F_1'] = df['F_0'] + LEARNING_RATE * df['h_1']
print(f"\nStep 4: Update the model predictions to F_1.")
print(f"F_1 = F_0 + {LEARNING_RATE} * h_1")


# (New) Step 2: Compute the new residuals for the updated model (F_1).
df['Residual_1'] = y - df['F_1']

print("\n--- Results for Round 1 ---")
print(df[['Size', 'Price', 'F_0', 'h_1', 'F_1', 'Residual_1']].round(2))

# Check how the error is shrinking
mse_after_round_1 = mean_squared_error(y, df['F_1'])
print(f"\nMean Squared Error (MSE) before this round: {initial_mse:.2f}")
print(f"Mean Squared Error (MSE) after this round:  {mse_after_round_1:.2f}")
print("The residuals are shrinking!")
print("-" * 50, "\n")


--- Round 1: Adding Weak Learner 1 (h_1) ---

Step 3: Fit a decision stump (h_1) to the residuals from Round 0.
The stump has learned to predict the following error corrections:
    h_1
0 -19.5
1 -19.5
2 -19.5
3 -19.5
4 -19.5
5 -19.5
6  78.0
7  78.0
8 -19.5
9 -19.5

Step 4: Update the model predictions to F_1.
F_1 = F_0 + 0.8 * h_1

--- Results for Round 1 ---
   Size  Price    F_0   h_1    F_1  Residual_1
0  1400    245  286.5 -19.5  270.9       -25.9
1  1600    312  286.5 -19.5  270.9        41.1
2  1700    279  286.5 -19.5  270.9         8.1
3  1875    308  286.5 -19.5  270.9        37.1
4  1100    199  286.5 -19.5  270.9       -71.9
5  1550    219  286.5 -19.5  270.9       -51.9
6  2350    405  286.5  78.0  348.9        56.1
7  2450    324  286.5  78.0  348.9       -24.9
8  1425    319  286.5 -19.5  270.9        48.1
9  1700    255  286.5 -19.5  270.9       -15.9

Mean Squared Error (MSE) before this round: 3260.05
Mean Squared Error (MSE) after this round:  1799.89
The residuals a

In [3]:


# ####################################################################
# --- Round 2: Adding a Second Weak Learner (h_2) ---
# ####################################################################
print(f"--- Round 2: Adding Weak Learner 2 (h_2) ---")

# Step 3: Fit a new weak learner to the residuals from the PREVIOUS round (Round 1).
print(f"\nStep 3: Fit a new decision stump (h_2) to the residuals from Round 1.")
stump_2 = DecisionTreeRegressor(max_depth=1, random_state=42)
stump_2.fit(X, df['Residual_1'])

# Predict the new error corrections
df['h_2'] = stump_2.predict(X)
print("The new stump has learned to predict the following error corrections:")
print(df[['h_2']].round(2))

# Step 4: Update the model's predictions again to F_2.
# F_2(x) = F_1(x) + learning_rate * h_2(x)
df['F_2'] = df['F_1'] + LEARNING_RATE * df['h_2']
print(f"\nStep 4: Update the model predictions to F_2.")
print(f"F_2 = F_1 + {LEARNING_RATE} * h_2")

# (New) Step 2: Compute the final residuals for the F_2 model.
df['Residual_2'] = y - df['F_2']

print("\n--- Results for Round 2 ---")
print(df[['Size', 'Price', 'F_1', 'h_2', 'F_2', 'Residual_2']].round(2))

# Check the error shrinkage again
mse_after_round_2 = mean_squared_error(y, df['F_2'])
print(f"\nMean Squared Error (MSE) before this round: {mse_after_round_1:.2f}")
print(f"Mean Squared Error (MSE) after this round:  {mse_after_round_2:.2f}")
print("The residuals continue to shrink!")
print("-" * 50, "\n")

print("--- Final Model Predictions after 2 rounds ---")
print(df[['Size', 'Price', 'F_0', 'F_1', 'F_2']].round(2))
print("\nNotice how the predictions (F_0 -> F_1 -> F_2) get closer to the actual Price.")


--- Round 2: Adding Weak Learner 2 (h_2) ---

Step 3: Fit a new decision stump (h_2) to the residuals from Round 1.
The new stump has learned to predict the following error corrections:
     h_2
0 -48.90
1  12.23
2  12.23
3  12.23
4 -48.90
5  12.23
6  12.23
7  12.23
8  12.23
9  12.23

Step 4: Update the model predictions to F_2.
F_2 = F_1 + 0.8 * h_2

--- Results for Round 2 ---
   Size  Price    F_1    h_2     F_2  Residual_2
0  1400    245  270.9 -48.90  231.78       13.22
1  1600    312  270.9  12.23  280.68       31.32
2  1700    279  270.9  12.23  280.68       -1.68
3  1875    308  270.9  12.23  280.68       27.32
4  1100    199  270.9 -48.90  231.78      -32.78
5  1550    219  270.9  12.23  280.68      -61.68
6  2350    405  348.9  12.23  358.68       46.32
7  2450    324  348.9  12.23  358.68      -34.68
8  1425    319  270.9  12.23  280.68       38.32
9  1700    255  270.9  12.23  280.68      -25.68

Mean Squared Error (MSE) before this round: 1799.89
Mean Squared Error (MSE) a